# Full-data entity matching → `matching_results.csv` + `candidates.csv` (saved to Drive)

1. **Runtime → Change runtime type → High-RAM** if available (full data needs ~10-14 GB).
2. **Runtime → Run all.** Approve the Google Drive access prompt.
3. Outputs go to `MyDrive/ML-CHALLENGE-AMAZON/outputs/`.

If Colab disconnects, run all again: the trained matcher is saved to Drive and reused, so only the test stage re-runs. Delete `outputs/matcher_lgbm.txt` to retrain.

In [ ]:
!pip -q install polars pyarrow bm25s sparse_dot_topn jellyfish unidecode rapidfuzz lightgbm

In [ ]:
import os
DATA_DIR = os.environ.get('DATA_DIR', '/content/drive/MyDrive/ML-CHALLENGE-AMAZON')
OUT_DIR  = os.environ.get('OUT_DIR',  DATA_DIR + '/outputs')
SEP = ','              # use '\t' if the submission must be tab-separated
WRITE_CANDIDATES = True  # candidates.csv is large (~1-2 GB on full data)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab; using DATA_DIR =', DATA_DIR)
print(sorted(p for p in os.listdir(DATA_DIR) if p.endswith('.tsv')))

In [ ]:
# Copy the TSVs to local disk: reading 2 GB through the Drive mount is much slower
import shutil
LOCAL = '/content/data' if os.path.exists('/content') else DATA_DIR
if LOCAL != DATA_DIR:
    os.makedirs(LOCAL, exist_ok=True)
    for f in os.listdir(DATA_DIR):
        if f.startswith(('train_', 'test_')) and f.endswith('.tsv') and not os.path.exists(f'{LOCAL}/{f}'):
            shutil.copy(f'{DATA_DIR}/{f}', f'{LOCAL}/{f}'); print('copied', f)

## Pipeline source (from the repo, embedded so the notebook is self-contained)

In [ ]:
os.makedirs('src', exist_ok=True)
open('src/__init__.py', 'w').close()

In [ ]:
%%writefile src/fast_normalization.py
"""
Optimized drop-in replacement for `normalize_pipeline` (notebooks/kaggle_mirror.ipynb, cell 2).

Output columns and values are identical to the notebook version (verified by
benchmarks/run_benchmark.py --check-parity), except where an option explicitly says otherwise.

What changed vs. the notebook and why:
  * One pass per row instead of ~15 full-column passes (2x unidecode list-comps, 13 pandas
    `.str.replace` calls, 1 geo loop, 1 row-wise `DataFrame.apply`, 3 threshold list-comps).
  * `unidecode` is skipped for ASCII strings (`str.isascii()` is a C-level scan); it was called
    3x per address in the notebook (clean_address + once per comma chunk).
  * All regexes are pre-compiled at import time; the 9 suffix regexes only run when a cheap
    substring trigger is present.
  * Geo chunks are scanned right-to-left and the scan stops after 2 valid chunks (only
    chunk_1 / chunk_2 are ever used).
  * `geo_df.apply(assign_base_key, axis=1)` (builds a pandas Series per row) is replaced by
    plain tuple logic inside the same row pass.
  * Threshold passes use `pd.factorize` + `np.bincount` instead of `value_counts` + Python
    `re.search` on every key.
  * Row chunks are processed in a fork-based process pool (pure-Python work -> GIL-bound, so
    threads would not help). No Polars / BLAS threads are alive inside the workers.
  * `df.copy()` of the raw frame is gone; strings are stored as pyarrow strings (~3-5x less RAM
    than Python objects).
  * `joint_thresholds=True` (via `normalize_sources`) computes block-size thresholds on the union
    of S1+S2+S3 so the same address gets the same `geo_block_key` in every source. The notebook
    computes thresholds per source, so a city split into `<key>_<letter>` in S2 (5M rows) can stay
    unsplit in S1 (2.2M rows) and the two blocks never meet in geo blocking.
"""
from __future__ import annotations

import logging
import multiprocessing as mp
import os
import re
from typing import Iterable, Sequence

import jellyfish
import numpy as np
import pandas as pd
from unidecode import unidecode

log = logging.getLogger(__name__)

# ---------------------------------------------------------------------------------------------
# Constants (identical to the notebook)
# ---------------------------------------------------------------------------------------------
SUFFIX_MAP = {
    r'\bprivate limited\b': 'ltd', r'\bpvt ltd\b': 'ltd', r'\bpvt\.?\s*ltd\.?\b': 'ltd',
    r'\bcorporation\b': 'corp', r'\bincorporated\b': 'inc', r'\bstreet\b': 'st',
    r'\broad\b': 'rd', r'\bavenue\b': 'ave', r'\bboulevard\b': 'blvd'
}
COMPILED_SUFFIXES = [(re.compile(p), r) for p, r in SUFFIX_MAP.items()]
# Every suffix pattern contains one of these literals; if none is present no pattern can match.
_SUFFIX_TRIGGERS = ("private", "pvt", "corporation", "incorporated", "street", "road",
                    "avenue", "boulevard")

GEO_STOPWORDS = frozenset({
    "drive", "dr", "street", "st", "road", "rd", "avenue", "ave", "lane", "ln",
    "boulevard", "blvd", "no", "number", "plot", "flat", "shop", "floor", "building",
    "bldg", "near", "opp", "opposite", "behind", "city", "town", "pradesh", "state",
    "ltd", "inc", "corp", "co", "llc", "room", "suite", "ste", "apt", "apartment",
    "marg", "nagar", "vihar", "puram", "gali", "bhavan", "mahal", "complex",
    "bagh", "chowk", "naka", "taluka", "zila", "mandal", "gram", "phase", "sector",
    "khand", "colony", "enclave", "extension", "ext", "cross", "main",
    "rue", "chemin", "place", "allee", "route", "impasse", "batiment",
    "immeuble", "etage", "cedex", "bp", "boite", "postale"
})
_COUNTRY_ALIASES = frozenset({"usa", "us", "india", "ind", "france"})

_RE_PUNCT = re.compile(r'[^\w\s]')
_RE_WS = re.compile(r'\s+')
_RE_PIN = re.compile(r'\b\d{5,6}\b(?=[^\w]*$|\s*[a-z]{2,}\s*$)')
_RE_PIN_ANY = re.compile(r'\b\d{5,6}\b')
_RE_ENDS_PIN = re.compile(r'\d{5,6}$')


def _is_missing(x) -> bool:
    # Equivalent to `not pd.notna(x)` for scalars, ~10x cheaper.
    return x is None or (isinstance(x, float) and x != x) or x is pd.NA or x is pd.NaT


def _ascii(s: str) -> str:
    return s if s.isascii() else unidecode(s)


def _clean_name(x) -> str:
    if _is_missing(x):
        return ""
    s = _RE_PUNCT.sub(' ', _ascii(str(x)).lower())
    if any(t in s for t in _SUFFIX_TRIGGERS):
        for pat, repl in COMPILED_SUFFIXES:
            s = pat.sub(repl, s)
    return _RE_WS.sub(' ', s).strip()


def _clean_address(x) -> str:
    if _is_missing(x):
        return ""
    return _RE_WS.sub(' ', _RE_PUNCT.sub(' ', _ascii(str(x)).lower())).strip()


def _geo_parts(addr, country) -> tuple[str, str | None, str | None, str | None]:
    """Returns (country_str, pin, chunk_1, chunk_2) exactly as the notebook's geo loop."""
    c_str = str(country).lower().strip() if not _is_missing(country) else ""
    r_addr = str(addr).lower().strip() if not _is_missing(addr) else ""

    m = _RE_PIN.search(r_addr)
    pin = m.group(0) if m else None

    found: list[str] = []
    # Right-to-left: only the last two valid chunks are ever used.
    for chunk in reversed(r_addr.split(',')):
        chunk = chunk.strip()
        if not chunk:
            continue
        chunk = _ascii(chunk)
        cleaned = _RE_PIN_ANY.sub('', chunk)
        cleaned = _RE_PUNCT.sub(' ', cleaned)
        cleaned = _RE_WS.sub(' ', cleaned).strip()
        if (len(cleaned) > 1 and cleaned not in GEO_STOPWORDS
                and cleaned != c_str and cleaned not in _COUNTRY_ALIASES):
            found.append(cleaned.replace(" ", "_"))
            if len(found) == 2:
                break
    c1 = found[0] if found else None
    c2 = found[1] if len(found) > 1 else None
    return c_str, pin, c1, c2


def _process_rows(args: tuple[list, list, list]) -> tuple[list, ...]:
    """Worker: one pass over a chunk of rows. Pure Python; runs in a child process."""
    names, addrs, countries = args
    n = len(names)
    clean_name = [""] * n
    clean_addr = [""] * n
    phon = [""] * n
    base_key = [""] * n
    c_strs = [""] * n
    c1s: list = [None] * n
    c2s: list = [None] * n
    metaphone = jellyfish.metaphone
    for i in range(n):
        cn = _clean_name(names[i])
        clean_name[i] = cn
        clean_addr[i] = _clean_address(addrs[i])
        phon[i] = metaphone(cn) if cn else ""
        c_str, pin, c1, c2 = _geo_parts(addrs[i], countries[i])
        c_strs[i], c1s[i], c2s[i] = c_str, c1, c2
        if pin:
            base_key[i] = f"{c_str}_{pin}"
        elif c1:
            base_key[i] = f"{c_str}_{c1}"
        else:
            base_key[i] = f"{c_str}_unknown"
    return clean_name, clean_addr, phon, base_key, c_strs, c1s, c2s


def _chunks(n: int, size: int) -> Iterable[tuple[int, int]]:
    for i in range(0, n, size):
        yield i, min(i + size, n)


def _row_pass(df: pd.DataFrame, n_jobs: int, chunk_rows: int) -> dict[str, list]:
    names = df['business_name'].tolist()
    addrs = df['business_address'].tolist()
    countries = df['country'].tolist()
    tasks = [(names[a:b], addrs[a:b], countries[a:b]) for a, b in _chunks(len(df), chunk_rows)]

    if n_jobs > 1 and len(tasks) > 1:
        ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
        with ctx.Pool(processes=n_jobs) as pool:
            parts = pool.map(_process_rows, tasks, chunksize=1)
    else:
        parts = [_process_rows(t) for t in tasks]
    del tasks, names, addrs, countries

    keys = ("clean_name", "clean_address", "phonetic_hash", "base_key", "c_str", "chunk_1", "chunk_2")
    out: dict[str, list] = {k: [] for k in keys}
    for p in parts:
        for k, v in zip(keys, p):
            out[k].extend(v)
    return out


def _apply_thresholds(
    base_key: np.ndarray, c_str: np.ndarray, c1: np.ndarray, c2: np.ndarray,
    first_char: np.ndarray, upper_threshold: int, lower_threshold: int,
) -> np.ndarray:
    """Vectorized version of the notebook's 3 threshold passes (same semantics)."""
    keys = base_key.astype(object)

    def counts_of(arr: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        codes, uniq = pd.factorize(arr)
        cnt = np.bincount(codes, minlength=len(uniq))
        return codes, uniq, cnt

    def pin_suffix(uniq) -> np.ndarray:
        return np.fromiter((bool(_RE_ENDS_PIN.search(k)) for k in uniq), bool, len(uniq))

    # Pass 1: massive non-PIN keys -> country_chunk2_chunk1 (only if chunk_2 exists)
    codes, uniq, cnt = counts_of(keys)
    split_key = (cnt > upper_threshold) & ~pin_suffix(uniq)
    rows = np.flatnonzero(split_key[codes] & (c2 != None))  # noqa: E711
    if len(rows):
        keys[rows] = [f"{c}_{b}_{a}" for c, a, b in zip(c_str[rows], c1[rows], c2[rows])]

    # Pass 2: still mega non-PIN keys -> append first letter of clean_name
    codes, uniq, cnt = counts_of(keys)
    split_key = (cnt > upper_threshold) & ~pin_suffix(uniq)
    rows = np.flatnonzero(split_key[codes])
    if len(rows):
        keys[rows] = [f"{k}_{f}" for k, f in zip(keys[rows], first_char[rows])]

    # Pass 3: tiny keys -> <country>_unknown
    codes, uniq, cnt = counts_of(keys)
    small = cnt < lower_threshold
    if small.any():
        repl = np.array([f"{k.split('_')[0]}_unknown" if s else k for k, s in zip(uniq, small)], dtype=object)
        keys = repl[codes]
    return keys


def _assemble(df: pd.DataFrame, cols: dict[str, list], geo_key: np.ndarray,
              string_dtype: str) -> pd.DataFrame:
    out = df.drop(columns=['business_name', 'business_address'])
    sd = pd.StringDtype(string_dtype) if string_dtype == "pyarrow" else object
    cn = pd.Series(cols["clean_name"], index=df.index, dtype=sd)
    ca = pd.Series(cols["clean_address"], index=df.index, dtype=sd)
    out['clean_name'] = cn
    out['clean_address'] = ca
    out['geo_block_key'] = pd.Series(geo_key, index=df.index, dtype=sd)
    out['phonetic_hash'] = pd.Series(cols["phonetic_hash"], index=df.index, dtype=sd)
    out['search_document'] = cn + " " + ca
    return out


def _prepare(df: pd.DataFrame, n_jobs: int | None, chunk_rows: int) -> dict:
    n_jobs = n_jobs or os.cpu_count() or 1
    cols = _row_pass(df, n_jobs, chunk_rows)
    return {
        "cols": cols,
        "base_key": np.array(cols.pop("base_key"), dtype=object),
        "c_str": np.array(cols.pop("c_str"), dtype=object),
        "c1": np.array(cols.pop("chunk_1"), dtype=object),
        "c2": np.array(cols.pop("chunk_2"), dtype=object),
        "first": np.array([n[0] if n else 'z' for n in cols["clean_name"]], dtype=object),
    }


def normalize_pipeline(
    df: pd.DataFrame,
    upper_threshold: int = 15000,
    lower_threshold: int = 50,
    n_jobs: int | None = None,
    chunk_rows: int = 100_000,
    string_dtype: str = "pyarrow",
) -> pd.DataFrame:
    """Drop-in for the notebook's `normalize_pipeline` (per-source thresholds, same output)."""
    log.info("Normalizing %d rows (n_jobs=%s)", len(df), n_jobs or os.cpu_count())
    p = _prepare(df, n_jobs, chunk_rows)
    geo_key = _apply_thresholds(p["base_key"], p["c_str"], p["c1"], p["c2"], p["first"],
                                upper_threshold, lower_threshold)
    return _assemble(df, p["cols"], geo_key, string_dtype)


def normalize_sources(
    sources: Sequence[pd.DataFrame],
    upper_threshold: int = 15000,
    lower_threshold: int = 50,
    joint_thresholds: bool = True,
    n_jobs: int | None = None,
    chunk_rows: int = 100_000,
    string_dtype: str = "pyarrow",
) -> list[pd.DataFrame]:
    """
    Normalize several sources. With joint_thresholds=True, block-size thresholds are computed on
    the union of all sources, so identical addresses get identical geo_block_key values in every
    source (recall fix, see module docstring). joint_thresholds=False == calling
    normalize_pipeline on each source independently (notebook behavior).
    """
    if not joint_thresholds:
        return [normalize_pipeline(df, upper_threshold, lower_threshold, n_jobs, chunk_rows, string_dtype)
                for df in sources]
    preps = [_prepare(df, n_jobs, chunk_rows) for df in sources]
    sizes = [len(df) for df in sources]
    cat = {k: np.concatenate([p[k] for p in preps]) for k in ("base_key", "c_str", "c1", "c2", "first")}
    geo_all = _apply_thresholds(cat["base_key"], cat["c_str"], cat["c1"], cat["c2"], cat["first"],
                                upper_threshold, lower_threshold)
    del cat
    bounds = np.cumsum([0] + sizes)
    return [_assemble(df, p["cols"], geo_all[bounds[i]:bounds[i + 1]], string_dtype)
            for i, (df, p) in enumerate(zip(sources, preps))]


In [ ]:
%%writefile src/fast_blocking.py
"""
Optimized candidate generation (replaces cell 4 of notebooks/kaggle_mirror.ipynb).

Key ideas (details and measurements in docs/blocking_optimization.md):

1. BM25 as sparse matrix multiplication.
   bm25s (numpy backend) scores every query by allocating a dense float array of size N_docs,
   filling it with np.add.at and arg-partitioning all N_docs entries. Cost per query is O(N_docs)
   no matter how selective the query is: 883k India S1 queries x 2M-doc index = ~1.8e12 element
   ops per index per modality. Here the index is a CSR matrix W^T (vocab x docs) holding exactly
   the bm25s "lucene" weights idf(t) * tf/(tf + k1*(1-b+b*dl/avgdl)), the queries are a CSR
   term-count matrix Q, and scores = Q @ W^T with per-row top-k done by sparse_dot_topn in C++
   across all cores. Cost per query = number of postings touched, output is O(k) per query,
   and only non-zero scores are returned (bm25s pads every query to k with random score-0 docs).

2. Tokenize once. Every document is tokenized once per modality with a Polars (Rust,
   multi-threaded) regex identical to bm25s' `(?u)\\b\\w\\w+\\b` + the same English stopword list,
   into one shared vocabulary per country. The notebook tokenized S2 twice (index + queries) and
   rebuilt a fresh vocabulary for every 50k-row query chunk (then bm25s converted ids -> strings
   -> index ids again).

3. Integer entity handles. Entities are int64 positions into one concatenated id array; pairs
   are int64 keys a*N+b. The pandas outer merges on Python-string columns are replaced by Polars
   joins on integers; ids are decoded to strings once at the end (optional).

4. Geo blocking in one sort-based pass. The notebook filtered the whole frame once per block
   (`s1_df[s1_df.geo_block_key == block]`, O(N x B)) and ran the pair loops 12/13/23 separately,
   vectorizing every document twice. Here rows are argsorted by block once, each block is
   char-n-gram counted once for all three sources, and per-target TF-IDF weights reproduce
   `TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4)).fit(target)` exactly. Blocks are
   distributed over a fork process pool (the n-gram analyzer is Python, i.e. GIL-bound).
"""
from __future__ import annotations

import logging
import math
import multiprocessing as mp
import os
import time
from dataclasses import dataclass, field
from typing import Sequence

import numpy as np
import pandas as pd
import polars as pl
import scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer
from sparse_dot_topn import sp_matmul_topn

try:  # the exact list bm25s.tokenize(stopwords="english") uses
    from bm25s.stopwords import STOPWORDS_EN as _BM25S_STOPWORDS
except Exception:  # pragma: no cover - fallback keeps the module importable without bm25s
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS as _BM25S_STOPWORDS

log = logging.getLogger(__name__)

TOKEN_PATTERN = r"\b\w\w+\b"  # Rust regex: \b and \w are Unicode-aware, same as Python (?u)
STOPWORDS = sorted(set(_BM25S_STOPWORDS))


@dataclass
class BlockingConfig:
    top_k: int = 7
    n_threads: int = field(default_factory=lambda: os.cpu_count() or 1)
    # Query S2 against S3 as well (the notebook does). The ground truth only contains S1->S2/S3
    # pairs, so for the leaderboard metric these candidates are pure overhead; set False to
    # skip ~45% of BM25 work and ~1/3 of geo work.
    include_s2_s3: bool = True
    # BM25 parameters (bm25s defaults: method="lucene", k1=1.5, b=0.75)
    k1: float = 1.5
    b: float = 0.75
    # Optional query-side pruning of very common tokens (df / N_target > max_df_ratio). Their
    # idf is ~0.1-1 so they barely change rankings but dominate postings traversed ("ltd",
    # "india", state names). None = exact bm25s scores. Measure recall before enabling.
    max_df_ratio: float | None = None
    query_chunk_rows: int = 250_000
    tokenize_chunk_rows: int = 1_000_000
    # Geo stage
    geo_ngram_range: tuple[int, int] = (2, 4)
    geo_n_jobs: int = field(default_factory=lambda: os.cpu_count() or 1)
    geo_blocks_per_task: int = 64
    # Output
    decode_ids: bool = True


# ---------------------------------------------------------------------------------------------
# Entity table
# ---------------------------------------------------------------------------------------------
@dataclass
class EntityTable:
    """All sources concatenated once. Row position == global integer handle."""
    ids: np.ndarray                 # object/str, entity_id per handle
    source: np.ndarray              # int8: 0=S1, 1=S2, 2=S3
    country: np.ndarray             # int32 codes into country_names, -1 = NaN country
    country_names: list[str]
    text: pl.Series                 # search_document
    phonetic: pl.Series             # phonetic_hash
    geo_key: np.ndarray             # object, geo_block_key

    @classmethod
    def build(cls, sources: Sequence[pd.DataFrame]) -> "EntityTable":
        def col(name: str) -> list[pd.Series]:
            return [s[name] for s in sources]

        def to_pl(series: list[pd.Series]) -> pl.Series:
            return pl.concat([pl.Series(s.fillna("").astype(str).to_numpy(dtype=object), dtype=pl.String)
                              if s.dtype == object else pl.from_pandas(s.fillna("")).cast(pl.String)
                              for s in series])

        ids = np.concatenate([s.to_numpy(dtype=object) for s in col('entity_id')])
        source = np.concatenate([np.full(len(s), i, np.int8) for i, s in enumerate(sources)])
        # notebook: s['country'].str.lower(), NaN rows excluded from the BM25 loop
        low = pd.concat([s.astype(object) for s in col('country')], ignore_index=True).str.lower()
        codes, uniq = pd.factorize(low, sort=True)
        return cls(ids=ids, source=source, country=codes.astype(np.int32), country_names=list(uniq),
                   text=to_pl(col('search_document')), phonetic=to_pl(col('phonetic_hash')),
                   geo_key=pd.concat(col('geo_block_key'), ignore_index=True).astype(object).to_numpy())

    def __len__(self) -> int:
        return len(self.ids)


# ---------------------------------------------------------------------------------------------
# Tokenization -> sparse term-count matrices over a shared vocabulary
# ---------------------------------------------------------------------------------------------
class _Vocab:
    def __init__(self) -> None:
        self.table = pl.DataFrame(schema={"t": pl.String, "tid": pl.UInt32})

    def __len__(self) -> int:
        return self.table.height

    def encode(self, tokens: pl.DataFrame) -> pl.DataFrame:
        # maintain_order: token ids (hence tie-breaking order in top-k) must not depend on thread timing
        new = tokens.select("t").unique(maintain_order=True).join(self.table, on="t", how="anti", maintain_order="left")
        if new.height:
            start = self.table.height
            new = new.with_columns(pl.int_range(start, start + new.height, dtype=pl.UInt32).alias("tid"))
            self.table = pl.concat([self.table, new])
        return tokens.join(self.table, on="t", how="inner")


def _term_counts(texts: pl.Series, vocab: _Vocab, chunk_rows: int) -> sp.csr_matrix:
    """CSR (n_docs x |vocab| at call time) of token counts; bm25s.tokenize semantics."""
    n = len(texts)
    rows, cols = [], []
    for start in range(0, n, chunk_rows):
        part = texts.slice(start, chunk_rows)
        toks = (
            pl.DataFrame({"t": part})
            .with_row_index("d")
            .with_columns(pl.col("t").str.to_lowercase().str.extract_all(TOKEN_PATTERN))
            .explode("t")
            .drop_nulls("t")
            .filter(~pl.col("t").is_in(STOPWORDS))
        )
        enc = vocab.encode(toks)
        rows.append(enc["d"].to_numpy().astype(np.int64) + start)
        cols.append(enc["tid"].to_numpy())
    r = np.concatenate(rows) if rows else np.empty(0, np.int64)
    c = np.concatenate(cols) if cols else np.empty(0, np.uint32)
    m = sp.csr_matrix((np.ones(len(r), np.float32), (r, c.astype(np.int64))), shape=(n, max(len(vocab), 1)))
    m.sum_duplicates()
    return m


def _pad_cols(m: sp.csr_matrix, n_cols: int) -> sp.csr_matrix:
    if m.shape[1] == n_cols:
        return m
    return sp.csr_matrix((m.data, m.indices, m.indptr), shape=(m.shape[0], n_cols))


def _bm25_index(tf: sp.csr_matrix, k1: float, b: float) -> tuple[sp.csr_matrix, np.ndarray]:
    """
    Returns (W^T as CSR vocab x docs, df) with bm25s 'lucene' weights:
      idf = ln(1 + (N - df + .5)/(df + .5)),  w = idf * tf / (tf + k1*(1 - b + b*dl/avgdl))
    dl counts duplicate tokens (len(token_ids) in bm25s); empty docs have dl=0.
    """
    n_docs = tf.shape[0]
    dl = np.asarray(tf.sum(axis=1)).ravel().astype(np.float32)
    avgdl = float(dl.mean()) if n_docs else 1.0
    df = np.bincount(tf.indices, minlength=tf.shape[1]).astype(np.float32)
    idf = np.zeros(tf.shape[1], np.float32)
    nz = df > 0
    idf[nz] = np.log1p((n_docs - df[nz] + 0.5) / (df[nz] + 0.5))
    row_of = np.repeat(np.arange(n_docs), np.diff(tf.indptr))
    t = tf.data
    w = idf[tf.indices] * (t / (t + k1 * ((1 - b) + b * dl[row_of] / max(avgdl, 1e-9))))
    W = sp.csr_matrix((w.astype(np.float32), tf.indices, tf.indptr), shape=tf.shape)
    return W.T.tocsr(), df


def _topk_pairs(Q: sp.csr_matrix, WT: sp.csr_matrix, q_handles: np.ndarray, t_handles: np.ndarray,
                cfg: BlockingConfig) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    out_a, out_b, out_s = [], [], []
    k = min(cfg.top_k, WT.shape[1])
    if k == 0 or Q.shape[0] == 0:
        return np.empty(0, np.int64), np.empty(0, np.int64), np.empty(0, np.float32)
    for start in range(0, Q.shape[0], cfg.query_chunk_rows):
        q = Q[start:start + cfg.query_chunk_rows]
        m = sp_matmul_topn(q, WT, top_n=k, n_threads=cfg.n_threads).tocoo()
        keep = m.data > 0
        out_a.append(q_handles[start + m.row[keep]])
        out_b.append(t_handles[m.col[keep]])
        out_s.append(m.data[keep].astype(np.float32))
    return np.concatenate(out_a), np.concatenate(out_b), np.concatenate(out_s)


def _pairs_frame(a: np.ndarray, b: np.ndarray, s: np.ndarray, n: int, col: str) -> pl.DataFrame:
    return pl.DataFrame({"key": a.astype(np.int64) * n + b.astype(np.int64), col: s})


def _scan_modality(ent: EntityTable, texts: pl.Series, groups: dict[int, np.ndarray],
                   col: str, cfg: BlockingConfig) -> pl.DataFrame:
    """One modality for one country. groups: source -> sorted global handles."""
    vocab = _Vocab()
    tf = {s: _term_counts(texts.gather(h), vocab, cfg.tokenize_chunk_rows) for s, h in groups.items() if len(h)}
    V = max(len(vocab), 1)
    tf = {s: _pad_cols(m, V) for s, m in tf.items()}

    jobs = [(0, 1), (0, 2)] + ([(1, 2)] if cfg.include_s2_s3 else [])
    frames = []
    for target in (1, 2):
        if target not in tf or not any(q in tf and t == target for q, t in jobs):
            continue
        WT, df = _bm25_index(tf[target], cfg.k1, cfg.b)
        drop = None
        if cfg.max_df_ratio is not None:
            drop = np.flatnonzero(df > cfg.max_df_ratio * tf[target].shape[0])
        for q, t in jobs:
            if t != target or q not in tf:
                continue
            Q = tf[q]
            if drop is not None and len(drop):
                Q = Q.copy()
                mask = np.isin(Q.indices, drop)
                Q.data[mask] = 0
                Q.eliminate_zeros()
            a, b, s = _topk_pairs(Q, WT, groups[q], groups[t], cfg)
            frames.append(_pairs_frame(a, b, s, len(ent), col))
        del WT
    if not frames:
        return pl.DataFrame(schema={"key": pl.Int64, col: pl.Float32})
    return pl.concat(frames)


def generate_global_candidate_pool(ent: EntityTable, cfg: BlockingConfig) -> pl.DataFrame:
    """Stages 1-3: BM25 text + phonetic per country. Returns key | bm25_text_score | bm25_phonetic_score."""
    log.info("BM25 candidate generation (threads=%d, top_k=%d)", cfg.n_threads, cfg.top_k)
    order = np.lexsort((ent.source, ent.country))          # one sort instead of 3 filters x countries
    cty, src = ent.country[order], ent.source[order]
    bounds = np.flatnonzero(np.diff(cty)) + 1
    per_country = []
    for seg in np.split(np.arange(len(order)), bounds):
        if not len(seg) or cty[seg[0]] < 0:                 # NaN country: excluded (as notebook)
            continue
        h = order[seg]
        groups = {s: np.sort(h[src[seg] == s]) for s in (0, 1, 2)}
        name = ent.country_names[cty[seg[0]]]
        t0 = time.perf_counter()
        text = _scan_modality(ent, ent.text, groups, "bm25_text_score", cfg)
        phon = _scan_modality(ent, ent.phonetic, groups, "bm25_phonetic_score", cfg)
        merged = text.join(phon, on="key", how="full", coalesce=True)
        per_country.append(merged)
        log.info("[%s] S1=%d S2=%d S3=%d -> %d pairs in %.2fs", name.upper(), len(groups[0]),
                 len(groups[1]), len(groups[2]), merged.height, time.perf_counter() - t0)
    if not per_country:
        return pl.DataFrame(schema={"key": pl.Int64, "bm25_text_score": pl.Float32,
                                    "bm25_phonetic_score": pl.Float32})
    return pl.concat(per_country).unique("key", keep="first").with_columns(_fill0(_BM25_COLS))


# ---------------------------------------------------------------------------------------------
# Geo blocking
# ---------------------------------------------------------------------------------------------
_GEO_DOCS: list[str] | None = None  # set only inside pool workers via initializer


def _geo_init(docs: list[str]) -> None:
    global _GEO_DOCS
    _GEO_DOCS = docs


def _l2_rows(m: sp.csr_matrix) -> sp.csr_matrix:
    sq = np.asarray(m.multiply(m).sum(axis=1)).ravel()
    inv = np.zeros_like(sq)
    nz = sq > 0
    inv[nz] = 1.0 / np.sqrt(sq[nz])
    return sp.csr_matrix(sp.diags(inv.astype(np.float32)) @ m)


def _geo_block(docs: list[str], handles: np.ndarray, src: np.ndarray, pairs: Sequence[tuple[int, int]],
               top_k: int, ngram_range: tuple[int, int]) -> list[np.ndarray]:
    """One block, all source pairs. Reproduces TfidfVectorizer(char_wb).fit(target)/transform(query)."""
    cv = CountVectorizer(analyzer="char_wb", ngram_range=ngram_range, dtype=np.float32)
    try:
        C = cv.fit_transform(docs).tocsr()
    except ValueError:  # empty vocabulary
        return []
    out = []
    for q, t in pairs:
        qi, ti = np.flatnonzero(src == q), np.flatnonzero(src == t)
        if not len(qi) or not len(ti):
            continue
        Ct, Cq = C[ti], C[qi]
        n = Ct.shape[0]
        df = np.bincount(Ct.indices, minlength=C.shape[1])
        idf = np.zeros(C.shape[1], np.float32)
        in_vocab = df > 0          # query n-grams unseen in the target are dropped by transform()
        idf[in_vocab] = np.log((1 + n) / (1 + df[in_vocab])) + 1
        D = sp.diags(idf)
        T = _l2_rows(sp.csr_matrix(Ct @ D))
        Qm = _l2_rows(sp.csr_matrix(Cq @ D))
        m = sp_matmul_topn(Qm, T.T.tocsr(), top_n=min(top_k, n), n_threads=1).tocoo()
        keep = m.data > 0
        out.append(np.stack([handles[qi[m.row[keep]]].astype(np.float64),
                             handles[ti[m.col[keep]]].astype(np.float64),
                             m.data[keep].astype(np.float64)]))
    return out


def _geo_task(args: tuple) -> np.ndarray:
    blocks, pairs, top_k, ngram_range = args
    res = []
    for handles, src in blocks:
        docs = [_GEO_DOCS[h] for h in handles]
        res.extend(_geo_block(docs, handles, src, pairs, top_k, ngram_range))
    return np.concatenate(res, axis=1) if res else np.empty((3, 0))


def run_multi_source_geo_blocking(ent: EntityTable, cfg: BlockingConfig) -> pl.DataFrame:
    """Stage 4: local char n-gram TF-IDF inside shared geo blocks. Returns key | geo_ngram_score."""
    t0 = time.perf_counter()
    pairs = [(0, 1), (0, 2)] + ([(1, 2)] if cfg.include_s2_s3 else [])
    valid = ~pd.Series(ent.geo_key).str.endswith("_unknown").to_numpy(dtype=bool)
    codes, _ = pd.factorize(ent.geo_key)
    idx = np.flatnonzero(valid)
    order = idx[np.argsort(codes[idx], kind="stable")]
    oc = codes[order]
    starts = np.r_[0, np.flatnonzero(np.diff(oc)) + 1]
    ends = np.r_[starts[1:], len(order)]

    blocks = []
    for s, e in zip(starts, ends):
        h = order[s:e]
        src = ent.source[h]
        present = set(np.unique(src).tolist())
        if any(q in present and t in present for q, t in pairs):
            blocks.append((h, src))
    log.info("Geo: %d shared blocks", len(blocks))
    if not blocks:
        return pl.DataFrame(schema={"key": pl.Int64, "geo_ngram_score": pl.Float32})

    # Balance tasks: biggest blocks first, round-robin over tasks.
    blocks.sort(key=lambda x: -len(x[0]))
    n_tasks = max(1, math.ceil(len(blocks) / cfg.geo_blocks_per_task))
    tasks = [([], pairs, cfg.top_k, cfg.geo_ngram_range) for _ in range(n_tasks)]
    for i, blk in enumerate(blocks):
        tasks[i % n_tasks][0].append(blk)

    docs = ent.text.to_list()
    if cfg.geo_n_jobs > 1 and n_tasks > 1:
        ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
        with ctx.Pool(cfg.geo_n_jobs, initializer=_geo_init, initargs=(docs,)) as pool:
            parts = pool.map(_geo_task, tasks, chunksize=1)
    else:
        _geo_init(docs)
        try:
            parts = [_geo_task(t) for t in tasks]
        finally:
            _geo_init(None)  # type: ignore[arg-type]
    del docs
    allp = np.concatenate(parts, axis=1)
    a, b, s = allp[0].astype(np.int64), allp[1].astype(np.int64), allp[2].astype(np.float32)
    out = _pairs_frame(a, b, s, len(ent), "geo_ngram_score").unique("key", keep="first")
    log.info("Geo: %d pairs in %.2fs", out.height, time.perf_counter() - t0)
    return out


# ---------------------------------------------------------------------------------------------
# Orchestration + merge
# ---------------------------------------------------------------------------------------------
_BM25_COLS = ("bm25_text_score", "bm25_phonetic_score")
_SCORE_COLS = _BM25_COLS + ("geo_ngram_score",)


def _fill0(cols: Sequence[str]) -> list[pl.Expr]:
    # Only score columns: a frame-wide fill_null(0.0) would up-cast the int64 pair key to f64.
    return [pl.col(c).fill_null(0.0) for c in cols]


def merge_pools(bm25: pl.DataFrame, geo: pl.DataFrame) -> pl.DataFrame:
    """Outer union of BM25 and geo candidates on the int64 pair key (Polars hash join)."""
    return bm25.join(geo, on="key", how="full", coalesce=True).with_columns(_fill0(_SCORE_COLS))


def build_candidate_pool(s1: pd.DataFrame, s2: pd.DataFrame, s3: pd.DataFrame,
                         cfg: BlockingConfig | None = None) -> pl.DataFrame:
    """
    Full candidate pool: entity_A | entity_B | bm25_text_score | bm25_phonetic_score | geo_ngram_score
    (same schema as `final_master_pool` in the notebook, as a Polars frame; `.to_pandas()` if needed).
    With cfg.decode_ids=False, entity_A/entity_B are int64 handles into EntityTable.ids.
    """
    cfg = cfg or BlockingConfig()
    ent = EntityTable.build([s1, s2, s3])
    bm25 = generate_global_candidate_pool(ent, cfg)
    geo = run_multi_source_geo_blocking(ent, cfg)
    pool = merge_pools(bm25, geo)
    n = len(ent)
    pool = pool.with_columns((pl.col("key") // n).alias("entity_A"), (pl.col("key") % n).alias("entity_B")).drop("key")
    if cfg.decode_ids:
        ids = pl.Series(ent.ids.astype(str))
        pool = pool.with_columns(
            ids.gather(pool["entity_A"]).alias("entity_A"), ids.gather(pool["entity_B"]).alias("entity_B"))
    log.info("Final candidate pool: %d pairs", pool.height)
    return pool.select("entity_A", "entity_B", "bm25_text_score", "bm25_phonetic_score", "geo_ngram_score")


In [ ]:
%%writefile src/match_pipeline.py
"""
End-to-end: raw TSVs -> candidate pool -> learned matcher -> matching_results.csv + candidates.csv.

    train:  normalize -> candidates -> pair features -> LightGBM (labels from train ground truth,
            validation split by S1 entity; picks the probability threshold that maximizes pair F1)
    test:   normalize -> candidates -> features -> predict -> threshold -> one row per S1 entity

Memory: pairs are kept as int handles; string features are computed in chunks.
"""
from __future__ import annotations

import gc
import logging
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

from src.fast_blocking import BlockingConfig, EntityTable, generate_global_candidate_pool, \
    merge_pools, run_multi_source_geo_blocking
from src.fast_normalization import normalize_sources

log = logging.getLogger(__name__)

FEATURES = [
    "bm25_text_score", "bm25_phonetic_score", "geo_ngram_score", "target_s3",
    "text_rank", "text_rel", "phon_rank", "phon_rel", "geo_rank", "n_cands",
    "name_ratio", "name_token_set", "name_partial", "name_jw", "addr_token_set", "addr_ratio",
    "same_phonetic", "same_geo_key", "len_diff",
]


@dataclass
class RunConfig:
    blocking: BlockingConfig = field(default_factory=lambda: BlockingConfig(include_s2_s3=False, decode_ids=False))
    joint_thresholds: bool = True
    feature_chunk: int = 2_000_000
    neg_sample: float = 0.25          # fraction of negative pairs kept for training
    val_frac: float = 0.2
    seed: int = 42


def read_sources(data_dir: str | Path, prefix: str) -> list[pd.DataFrame]:
    out = []
    for i in (1, 2, 3):
        p = Path(data_dir) / f"{prefix}_source{i}.tsv"
        t = time.perf_counter()
        df = pd.read_csv(p, sep="\t", dtype=str)
        log.info("read %s: %d rows in %.1fs", p.name, len(df), time.perf_counter() - t)
        out.append(df)
    return out


@dataclass
class Candidates:
    ent: EntityTable
    pool: pl.DataFrame      # a, b (int64 handles) + 3 scores
    name: pl.Series         # clean_name per handle
    addr: pl.Series         # clean_address per handle
    s1_ids: np.ndarray


def build_candidates(sources: list[pd.DataFrame], cfg: RunConfig) -> Candidates:
    s1_ids = sources[0]["entity_id"].to_numpy(dtype=object)
    norm = normalize_sources(sources, joint_thresholds=cfg.joint_thresholds)
    sources.clear()
    gc.collect()
    ent = EntityTable.build(norm)
    name = pl.concat([pl.from_pandas(n["clean_name"].astype(str)) for n in norm])
    addr = pl.concat([pl.from_pandas(n["clean_address"].astype(str)) for n in norm])
    del norm
    gc.collect()
    bm = generate_global_candidate_pool(ent, cfg.blocking)
    geo = run_multi_source_geo_blocking(ent, cfg.blocking)
    pool = merge_pools(bm, geo)
    del bm, geo
    n = len(ent)
    pool = pool.select((pl.col("key") // n).alias("a"), (pl.col("key") % n).alias("b"),
                       "bm25_text_score", "bm25_phonetic_score", "geo_ngram_score")
    # keep only S1-anchored pairs (the ground truth / submission are S1 -> S2/S3)
    src = pl.Series(ent.source)
    pool = pool.filter(src.gather(pool["a"]) == 0)
    log.info("candidate pairs (S1-anchored): %d", pool.height)
    return Candidates(ent, pool, name, addr, s1_ids)


def _rank_features(pool: pl.DataFrame, source: np.ndarray) -> pl.DataFrame:
    tgt = pl.Series(source).gather(pool["b"])
    p = pool.with_columns((tgt == 2).cast(pl.Float32).alias("target_s3"))
    grp = ["a", "target_s3"]
    return p.with_columns(
        pl.col("bm25_text_score").rank("min", descending=True).over(grp).cast(pl.Float32).alias("text_rank"),
        (pl.col("bm25_text_score") / pl.col("bm25_text_score").max().over(grp).clip(1e-6)).alias("text_rel"),
        pl.col("bm25_phonetic_score").rank("min", descending=True).over(grp).cast(pl.Float32).alias("phon_rank"),
        (pl.col("bm25_phonetic_score") / pl.col("bm25_phonetic_score").max().over(grp).clip(1e-6)).alias("phon_rel"),
        pl.col("geo_ngram_score").rank("min", descending=True).over(grp).cast(pl.Float32).alias("geo_rank"),
        pl.len().over(grp).cast(pl.Float32).alias("n_cands"),
    )


def pair_features(c: Candidates, pool: pl.DataFrame, chunk: int) -> np.ndarray:
    """float32 matrix [len(pool), len(FEATURES)], computed in chunks to bound RAM."""
    from rapidfuzz import fuzz, process
    from rapidfuzz.distance import JaroWinkler

    pool = _rank_features(pool, c.ent.source)
    phon, geo = c.ent.phonetic, pl.Series(c.ent.geo_key.astype(str))
    X = np.empty((pool.height, len(FEATURES)), np.float32)
    base_cols = FEATURES[:10]
    X[:, :10] = pool.select(base_cols).to_numpy().astype(np.float32)
    for s in range(0, pool.height, chunk):
        part = pool.slice(s, chunk)
        a, b = part["a"], part["b"]
        na, nb = c.name.gather(a).to_list(), c.name.gather(b).to_list()
        aa, ab = c.addr.gather(a).to_list(), c.addr.gather(b).to_list()
        e = s + part.height
        kw = dict(workers=-1, dtype=np.float32)
        X[s:e, 10] = process.cpdist(na, nb, scorer=fuzz.ratio, **kw)
        X[s:e, 11] = process.cpdist(na, nb, scorer=fuzz.token_set_ratio, **kw)
        X[s:e, 12] = process.cpdist(na, nb, scorer=fuzz.partial_ratio, **kw)
        X[s:e, 13] = process.cpdist(na, nb, scorer=JaroWinkler.normalized_similarity, **kw)
        X[s:e, 14] = process.cpdist(aa, ab, scorer=fuzz.token_set_ratio, **kw)
        X[s:e, 15] = process.cpdist(aa, ab, scorer=fuzz.ratio, **kw)
        X[s:e, 16] = (phon.gather(a) == phon.gather(b)).to_numpy().astype(np.float32)
        X[s:e, 17] = (geo.gather(a) == geo.gather(b)).to_numpy().astype(np.float32)
        X[s:e, 18] = np.abs(np.fromiter(map(len, na), np.float32, len(na)) - np.fromiter(map(len, nb), np.float32, len(nb)))
        del na, nb, aa, ab
        log.info("features %d/%d", e, pool.height)
    return X


def gt_pairs(gt: pd.DataFrame) -> pl.DataFrame:
    g = pl.from_pandas(gt[["source1_entity_id", "matched_entity_ids"]].dropna().astype(str))
    return (g.with_columns(pl.col("matched_entity_ids").str.split(",")).explode("matched_entity_ids")
             .select(pl.col("source1_entity_id").str.strip_chars().alias("A"),
                     pl.col("matched_entity_ids").str.strip_chars().alias("B"))
             .filter(pl.col("B") != "").unique())


def _labels(c: Candidates, pool: pl.DataFrame, gt: pd.DataFrame) -> tuple[np.ndarray, pl.DataFrame]:
    ids = pl.Series(c.ent.ids.astype(str))
    p = pool.select(ids.gather(pool["a"]).alias("A"), ids.gather(pool["b"]).alias("B"))
    tp = gt_pairs(gt).with_columns(pl.lit(1, pl.Int8).alias("y"))
    y = p.join(tp, on=["A", "B"], how="left", maintain_order="left")["y"].fill_null(0).to_numpy()
    return y, tp


def _best_threshold(p: np.ndarray, y: np.ndarray, n_true: int) -> tuple[float, float]:
    """Threshold maximizing pair-level F1; n_true includes true pairs the blocker missed."""
    order = np.argsort(-p)
    tp = np.cumsum(y[order])
    k = np.arange(1, len(p) + 1)
    f1 = 2 * tp / (k + n_true)
    i = int(np.argmax(f1))
    return float(p[order][i]), float(f1[i])


def train_matcher(c: Candidates, gt: pd.DataFrame, cfg: RunConfig):
    import lightgbm as lgb

    y, tp = _labels(c, c.pool, gt)
    rng = np.random.default_rng(cfg.seed)
    a = c.pool["a"].to_numpy()
    uniq_a = np.unique(a)
    val_a = rng.choice(uniq_a, int(len(uniq_a) * cfg.val_frac), replace=False)
    is_val = np.isin(a, val_a)
    keep = is_val | (y == 1) | (rng.random(len(y)) < cfg.neg_sample)
    idx = np.flatnonzero(keep)
    log.info("labels: %d positives in %d candidates (recall ceiling %.4f)", y.sum(), len(y), y.sum() / tp.height)
    X = pair_features(c, c.pool[idx], cfg.feature_chunk)
    yk, vk = y[idx], is_val[idx]
    w = np.where(yk == 1, 1.0, 1.0 / cfg.neg_sample).astype(np.float32)   # undo negative subsampling
    params = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
                  feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, seed=cfg.seed, verbose=-1)
    dtr = lgb.Dataset(X[~vk], yk[~vk], weight=w[~vk], feature_name=FEATURES)
    dva = lgb.Dataset(X[vk], yk[vk], feature_name=FEATURES, reference=dtr)
    model = lgb.train(params, dtr, num_boost_round=2000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)])
    # threshold on the validation S1 entities; denominator = all their true pairs
    pv = model.predict(X[vk], num_iteration=model.best_iteration)
    ids = pl.Series(c.ent.ids.astype(str))
    val_ids = set(ids.gather(pl.Series(val_a)).to_list())
    n_true_val = tp.filter(pl.col("A").is_in(list(val_ids))).height
    thr, f1 = _best_threshold(pv, yk[vk], n_true_val)
    log.info("validation: best threshold %.4f -> pair F1 %.4f (best_iter %d)", thr, f1, model.best_iteration)
    return model, thr, {"val_f1": f1, "threshold": thr, "best_iter": model.best_iteration,
                        "candidate_recall": float(y.sum() / tp.height)}


def predict(c: Candidates, model, thr: float, cfg: RunConfig) -> pl.DataFrame:
    X = pair_features(c, c.pool, cfg.feature_chunk)
    p = np.empty(len(X), np.float32)
    for s in range(0, len(X), cfg.feature_chunk):
        p[s:s + cfg.feature_chunk] = model.predict(X[s:s + cfg.feature_chunk], num_iteration=model.best_iteration)
    del X
    ids = pl.Series(c.ent.ids.astype(str))
    return c.pool.select(ids.gather(c.pool["a"]).alias("entity_A"), ids.gather(c.pool["b"]).alias("entity_B"),
                         "bm25_text_score", "bm25_phonetic_score", "geo_ngram_score").with_columns(
        pl.Series("match_prob", p))


def matching_results(scored: pl.DataFrame, s1_ids: np.ndarray, thr: float) -> pl.DataFrame:
    """Exactly one row per S1 entity (same rule as generate_baseline_submission)."""
    m = (scored.filter(pl.col("match_prob") >= thr).sort("match_prob", descending=True)
               .group_by("entity_A", maintain_order=True).agg(pl.col("entity_B").unique(maintain_order=True).str.join(",")))
    base = pl.DataFrame({"source1_entity_id": pd.Series(s1_ids).astype(str).tolist()})
    return base.join(m.rename({"entity_A": "source1_entity_id", "entity_B": "matched_entity_ids"}),
                     on="source1_entity_id", how="left", maintain_order="left").with_columns(
        pl.col("matched_entity_ids").fill_null(""))


def run_all(data_dir: str | Path, out_dir: str | Path, cfg: RunConfig | None = None,
            sep: str = ",", write_candidates: bool = True) -> dict:
    """train_* -> fit matcher; test_* -> matching_results.csv (+ candidates.csv) in out_dir."""
    cfg = cfg or RunConfig()
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()

    import json
    import lightgbm as lgb

    model_path, meta_path = out / "matcher_lgbm.txt", out / "matcher_meta.json"
    if model_path.exists() and meta_path.exists():      # resume after a disconnect: skip training
        log.info("=== TRAIN skipped: loading %s ===", model_path)
        model = lgb.Booster(model_file=str(model_path))
        info = json.loads(meta_path.read_text())
        thr = info["threshold"]
        model.best_iteration = info["best_iter"]
    else:
        log.info("=== TRAIN ===")
        gt = pd.read_csv(Path(data_dir) / "train_ground_truth.tsv", sep="\t", dtype=str)
        tr = build_candidates(read_sources(data_dir, "train"), cfg)
        model, thr, info = train_matcher(tr, gt, cfg)
        model.save_model(str(model_path), num_iteration=model.best_iteration)
        meta_path.write_text(json.dumps(info))
        del tr, gt
        gc.collect()

    log.info("=== TEST ===")
    te = build_candidates(read_sources(data_dir, "test"), cfg)
    scored = predict(te, model, thr, cfg)
    res = matching_results(scored, te.s1_ids, thr)
    res.write_csv(out / "matching_results.csv", separator=sep)
    if write_candidates:
        scored.write_csv(out / "candidates.csv", separator=sep, float_precision=5)
    info.update(test_s1=len(te.s1_ids), test_candidates=scored.height,
                s1_with_match=int((res["matched_entity_ids"] != "").sum()),
                runtime_min=round((time.perf_counter() - t0) / 60, 1))
    log.info("done: %s", info)
    return info


## Run: train matcher on `train_*` → predict `test_*` → write CSVs to Drive

In [ ]:
import sys, logging, importlib
sys.path.insert(0, os.getcwd())
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(message)s', force=True)
for m in ('src.fast_normalization', 'src.fast_blocking', 'src.match_pipeline'):
    if m in sys.modules: importlib.reload(sys.modules[m])
from src.match_pipeline import run_all, RunConfig
info = run_all(LOCAL, OUT_DIR, RunConfig(), sep=SEP, write_candidates=WRITE_CANDIDATES)
info

In [ ]:
import polars as pl
res = pl.read_csv(f'{OUT_DIR}/matching_results.csv', separator=SEP)
print(res.shape); res.head(10)